# 03 · Data cleaning and preprocessing
**Step 5:** handle missing values, duplicates, types, dates, categorical values, invalid records
and column names. Every action is recorded in a cleaning log (`reports/cleaning_log.csv`).

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
sns.set_theme(style="whitegrid", palette="viridis")

# Where the raw CSVs live. To rehearse on generated test data instead, use:
#   RAW_DIR = config.SYNTHETIC_DIR   (run `python scripts/make_synthetic_data.py` first)
RAW_DIR = config.RAW_DIR

In [ ]:
from src.data_loader import load_raw_tables
from src.cleaning import clean_all, save_cleaned

raw = load_raw_tables(RAW_DIR)
cleaned, log = clean_all(raw)
display(log)

## Rows before vs after cleaning

In [ ]:
display(pd.DataFrame({"raw_rows": {t: len(df) for t, df in raw.items()},
                      "clean_rows": {t: len(df) for t, df in cleaned.items()}})
        .assign(removed=lambda d: d.raw_rows - d.clean_rows))

## Validate the cleaned data

In [ ]:
checks = []
for t, pk in config.PRIMARY_KEYS.items():
    checks.append((f"{t}: primary key unique", not cleaned[t].duplicated(subset=pk).any()))
for child, ccol, parent, pcol in config.FOREIGN_KEYS:
    checks.append((f"{child}.{ccol} has a parent in {parent}",
                   bool(cleaned[child][ccol].isin(set(cleaned[parent][pcol])).all())))
checks.append(("review scores within 1-5", bool(cleaned["order_reviews"]["review_score"].between(1, 5).all())))
checks.append(("item prices positive", bool((cleaned["order_items"]["price"] > 0).all())))
display(pd.DataFrame(checks, columns=["check", "passed"]))

## Save cleaned tables to `data/cleaned/`

In [ ]:
save_cleaned(cleaned, log)
print("Saved to", config.CLEAN_DIR)